In [1]:
import torch
import lightning.pytorch as pl
from lightning.pytorch import loggers as pl_loggers
from lightning.pytorch.callbacks import LearningRateMonitor, ModelCheckpoint

from models.vqvae import VQVAE
from models.vqvae_2 import VQVAE2
from models.vqvae_finetune import VQVAE2FineTune
from dataset.vae_datamodule import VAEDataModule

In [3]:
def train():

    torch.set_float32_matmul_precision("medium")

    # model = VQVAE(in_channels=5)
    model = VQVAE2(
        in_channels=5,
        lr=5e-4,
        latent_dim=4,
        codebook_size=8192,
    )

    # model = VQVAE.load_from_checkpoint("logs/vqvae_5channel_full_ds_64_128_256_512_c1024/version_0/checkpoints/epoch=5-step=27111.ckpt", in_channels=5)

    dm = VAEDataModule(
        train_path="train.memmap",
        val_path="val.memmap",
        test_path="test.memmap",
        img_channel=5,
        img_res=(128, 64),
        batch_size=32,
        num_workers=0,
    )

    tb_logger = pl_loggers.TensorBoardLogger(
        save_dir="logs/", name="vqvae_dim4_c8192")

    lr_monitor = LearningRateMonitor(logging_interval="step")

    trainer = pl.Trainer(
        logger=tb_logger,
        max_epochs=12,
        callbacks=[lr_monitor],
    )

    trainer.fit(model, dm)


train()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Data preparation complete.
Data setup complete.



   | Name                | Type       | Params | Mode  | In sizes         | Out sizes       
--------------------------------------------------------------------------------------------------
0  | encoder_conv_in     | Conv2d     | 1.5 K  | train | [1, 5, 128, 64]  | [1, 32, 128, 64]
1  | encoder_down_blocks | ModuleList | 1.9 M  | train | ?                | ?               
2  | encoder_mid_blocks  | ModuleList | 1.1 M  | train | ?                | ?               
3  | encoder_norm_out    | GroupNorm  | 256    | train | [1, 128, 16, 8]  | [1, 128, 16, 8] 
4  | encoder_conv_out    | Conv2d     | 4.6 K  | train | [1, 128, 16, 8]  | [1, 4, 16, 8]   
5  | pre_quant_conv      | Conv2d     | 20     | train | [1, 4, 16, 8]    | [1, 4, 16, 8]   
6  | embedding           | Embedding  | 32.8 K | train | ?                | ?               
7  | post_quant_conv     | Conv2d     | 20     | train | [1, 4, 16, 8]    | [1, 4, 16, 8]   
8  | decoder_conv_in     | Conv2d     | 4.7 K  | train | [1, 4,

Epoch 11: 100%|██████████| 2301/2301 [05:47<00:00,  6.62it/s, v_num=0]     

`Trainer.fit` stopped: `max_epochs=12` reached.


Epoch 11: 100%|██████████| 2301/2301 [05:48<00:00,  6.61it/s, v_num=0]


In [3]:
def finetune():

    vqvae = VQVAE2.load_from_checkpoint("logs/vqvae_new/version_6/checkpoints/epoch=4-step=11505.ckpt", in_channels=5, lr=5e-3)
    model = VQVAE2FineTune(vqvae, lr=1e-4)

    dm = VAEDataModule(
        train_path="train.memmap",
        val_path="val.memmap",
        test_path="test.memmap",
        img_channel=5,
        img_res=(128, 64),
        batch_size=32,
        num_workers=0,
    )

    tb_logger = pl_loggers.TensorBoardLogger(save_dir="logs/", name="vqvae_new_finetune")

    lr_monitor = LearningRateMonitor(logging_interval="step")

    trainer = pl.Trainer(
        logger=tb_logger,
        max_epochs=5,
        callbacks=[lr_monitor],
    )

    trainer.fit(model, dm)

finetune()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3070') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


Data preparation complete.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Data setup complete.



  | Name          | Type                  | Params | Mode  | In sizes        | Out sizes                            
--------------------------------------------------------------------------------------------------------------------------
0 | vqvae         | VQVAE2                | 5.5 M  | train | [1, 5, 128, 64] | [[1, 5, 128, 64], [1, 4, 16, 8], '?']
1 | discriminator | PatchGanDiscriminator | 401 K  | train | ?               | ?                                    
--------------------------------------------------------------------------------------------------------------------------
2.9 M     Trainable params
3.0 M     Non-trainable params
5.9 M     Total params
23.759    Total estimated model params size (MB)
219       Modules in train mode
0         Modules in eval mode


...

c:\Users\hendr\miniconda3\envs\LUN\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


c:\Users\hendr\miniconda3\envs\LUN\Lib\site-packages\torch\nn\modules\conv.py:549: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\Convolution.cpp:1037.)
  return F.conv2d(
c:\Users\hendr\miniconda3\envs\LUN\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Epoch 1:  43%|████▎     | 1000/2301 [01:40<02:11,  9.90it/s, v_num=7]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined